# ML Lab 10

**Aim:** Design and implement a Convolutional Neural Network (CNN) using TensorFlow/Keras for classification and evaluate its performance using appropriate metrics.

## Experiment 1: Data Preprocessing and Reshaping
a) Load the dataset, encode labels, scale features, and reshape data for 1D CNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

# Load Iris data from sklearn (avoids file path dependency)
iris = load_iris()
data = pd.DataFrame(
    iris.data,
    columns=["sepal_length", "sepal_width", "petal_length", "petal_width"]
)
data["species"] = iris.target

X = data.drop("species", axis=1).values
y = to_categorical(data["species"].values, num_classes=3)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=data["species"]
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for Conv1D: (samples, features, channels)
X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("Preprocessing and reshaping complete.")
print(f"X_train_reshaped shape: {X_train_reshaped.shape}")
print(f"X_test_reshaped shape: {X_test_reshaped.shape}")

## Experiment 2: CNN Implementation
b) Design and train a 1D Convolutional Neural Network (CNN)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense

model = Sequential([
    Input(shape=(4, 1)),
    Conv1D(filters=32, kernel_size=2, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    X_train_reshaped, y_train,
    epochs=60,
    batch_size=8,
    validation_split=0.2,
    verbose=0
)

print("CNN training complete.")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

## Experiment 3: Performance Evaluation
c) Evaluate the model using Accuracy, Confusion Matrix, and Classification Report

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred_prob = model.predict(X_test_reshaped, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=iris.target_names))

## Experiment 4: Visualization
d) Visualize the training progress (accuracy and loss) and confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training accuracy
axes[0].plot(history.history["accuracy"], label="Train Accuracy", color="steelblue")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy", color="darkorange")
axes[0].set_title("CNN Accuracy Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Training loss
axes[1].plot(history.history["loss"], label="Train Loss", color="seagreen")
axes[1].plot(history.history["val_loss"], label="Val Loss", color="firebrick")
axes[1].set_title("CNN Loss Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names,
    ax=axes[2]
)
axes[2].set_title("CNN Confusion Matrix")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## Answers

1. **How do CNNs help in classification tasks?**  
   CNNs automatically learn local feature patterns through convolution filters, which improves classification performance on structured input data.

2. **Why are pooling layers used in CNN architecture?**  
   Pooling layers reduce dimensionality, lower computation, and make feature representations more robust to small input variations.

3. **Why reshape tabular input to `(samples, features, 1)` for Conv1D?**  
   Conv1D expects a 3D tensor where the last dimension is channels, so reshaping enables filters to slide across the feature sequence.